In [ ]:
# --- repo bootstrap: makes this notebook runnable from anywhere ---
import os, sys
ROOT = os.getcwd()
while ROOT != os.path.dirname(ROOT) and not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)
if not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = "/kaggle/input/datasets/sinadaneshgar/edgedeid"  # Kaggle dataset fallback
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("repo root:", ROOT)

In [ ]:
import sys

from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from tqdm import tqdm

from edgedeid.data.preprocessing import preprocess_text
from pseudolabel_utils import (
    load_dataset, clean_responses,
    deanonymize_dataframe_v1, deanonymize_dataframe_v2,
    run_model,
    handle_xml_merging, verify_name_variations,
    annotate_text_with_xml,
)

In [ ]:
PSEUDO_LABEL_MODEL = "SFT/Qwen3-1.7B-Base_SFT"
DATASET_NAME = "deepseek"  # "deepseek" | "haiku" | "hermes"

## 1 · Data Loading

In [ ]:
final_df = load_dataset(DATASET_NAME)
final_df = clean_responses(final_df)
print(f"Loaded {len(final_df):,} rows")

## 2 · Preprocessing

In [ ]:
final_df = preprocess_text(final_df, input_col="default_response_0", output_col="default_response_1")

## 3 · Deanonymization

In [ ]:
final_df = deanonymize_dataframe_v1(final_df, "default_response_1", "default_response_1_deanonymized")
final_df = deanonymize_dataframe_v2(final_df, "default_response_1", "xml_haiku", "native_text")
final_df = final_df[["native_text", "xml_haiku", "report_length", "features_to_include"]]
final_df.head()

## 4 · Model Inference

Start the vLLM server before running the cells below:
```bash
CUDA_VISIBLE_DEVICES="3" python -m vllm.entrypoints.openai.api_server \
    --model ../SFT/Qwen3-1.7B-Base_SFT --port 9995 --tensor-parallel-size 1
```

In [ ]:
client = OpenAI(api_key="EMPTY", base_url="http://localhost:9995/v1")

# Sanity check on a single note
text = "Bardia Khosravi 22yo fue al Hospital San Rafael el 22 de mayo de 2024 para ser atendida por el Dr. Gómez debido a su lesión cutánea."
print(run_model(text, client, PSEUDO_LABEL_MODEL))

In [ ]:
final_df["xml_pseudolabel"] = ""

def process_row(i):
    return i, run_model(final_df.at[i, "native_text"], client, PSEUDO_LABEL_MODEL)

with ThreadPoolExecutor(max_workers=100) as executor:
    futures = [executor.submit(process_row, i) for i in range(len(final_df))]
    for future in tqdm(as_completed(futures), total=len(final_df), desc="Processing rows"):
        i, result = future.result()
        final_df.at[i, "xml_pseudolabel"] = result

In [ ]:
exact_matches_indices = [
    i for i in range(len(final_df))
    if final_df.at[i, "xml_pseudolabel"] == final_df.at[i, "xml_haiku"]
]
print(f"Exact matches: {len(exact_matches_indices)}")
print(f"Non-matches:   {len(final_df) - len(exact_matches_indices)}")

final_df[final_df["xml_pseudolabel"].str.startswith("<phi_entities>\n<names/>")]

## 5 · XML Merging

In [ ]:
final_df["merged_xml"] = handle_xml_merging(final_df)

for i in tqdm(range(len(final_df))):
    final_df.at[i, "merged_xml"] = verify_name_variations(
        final_df.at[i, "native_text"], final_df.at[i, "merged_xml"]
    )

## 6 · Text Annotation

In [ ]:
annotated_texts = [
    annotate_text_with_xml(row["native_text"], row["merged_xml"])
    for _, row in tqdm(final_df.iterrows(), total=final_df.shape[0])
]
final_df["annotated_text"] = annotated_texts

## 7 · Save Results

In [ ]:
final_df = final_df[["native_text", "annotated_text", "report_length", "features_to_include"]]
final_df.rename(columns={"native_text": "default_response_0", "annotated_text": "default_response_1"}, inplace=True)
final_df

In [ ]:
output_paths = {
    "hermes":   "data/hermes/hermes_pseudolabeled.csv",
    "haiku":    "data/haiku/haiku_pseudolabeled.csv",
    "deepseek": "data/deepseek/deepseek_pseudolabeled.csv",
}
final_df.to_csv(output_paths[DATASET_NAME], index=False)